In [1]:
!pip install torch torchvision torchaudio --quiet

In [2]:
!pip install numpy pandas scikit-learn matplotlib --quiet

In [3]:
import os
import csv
import math
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")


In [4]:
DATA_PATH  = "clean_daily_close.csv"              # adjust path as needed
OUTPUT_CSV = "standalone_transformer_results.csv"
OUTPUT_TXT = "standalone_transformer_summary.txt"
TARGET_COL = "Close"

# ── Transformer Hyperparameters ─────────────────────────────────────────────
T_SEQ_LEN    = 96
T_LABEL_LEN  = 48
T_PRED_LEN   = 1

T_D_MODEL    = 256
T_N_HEADS    = 8
T_E_LAYERS   = 2      # encoder layers
T_D_LAYERS   = 1      # decoder layers

T_D_FF       = 1024
T_DROPOUT    = 0.1
T_ACTIVATION = "relu"

T_BATCH      = 32
T_EPOCHS     = 100
T_LR         = 0.0001
T_PATIENCE   = 10     # early stopping patience

# ── Experiment settings ──────────────────────────────────────────────────────
N_EXPERIMENTS      = 50
ROLLING_VOL_WINDOW = 30
MULTISTEP_HORIZONS = [5, 21]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def set_seed(seed: int):
    """Fully deterministic seeding for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [6]:
def split_data(df: pd.DataFrame):
    """Strict chronological split: Train ≤2022 | Val last-10% of Train | Test ≥2023."""
    df = df.sort_values("Date").reset_index(drop=True)
    train_mask    = df["Date"].dt.year <= 2022
    test_mask     = df["Date"].dt.year >= 2023
    df_train_full = df[train_mask].reset_index(drop=True)
    df_test       = df[test_mask].reset_index(drop=True)
    val_size      = int(len(df_train_full) * 0.10)
    df_train      = df_train_full.iloc[:-val_size].reset_index(drop=True)
    df_val        = df_train_full.iloc[-val_size:].reset_index(drop=True)
    return df_train, df_val, df_test

In [7]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return MAE, RMSE, MAPE, R² computed on original-scale arrays."""
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    mae    = float(mean_absolute_error(y_true, y_pred))
    rmse   = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask   = np.abs(y_true) > 1e-8
    mape   = float(np.mean(np.abs(
        (y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    r2     = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

In [8]:
def create_transformer_windows(series: np.ndarray,
                                seq_len: int, label_len: int, pred_len: int):
    """
    Sliding-window generator for Transformer (pred_len=1, no leakage).

    Encoder : series[i : i+seq_len]                              → (seq_len, 1)
    Decoder : series[i+seq_len-label_len : i+seq_len] + zeros(1) → (label_len+1, 1)
    Target  : series[i+seq_len]                                  → scalar

    Returns
    -------
    enc_x   : (N, seq_len,              1)  float32
    dec_x   : (N, label_len + pred_len, 1)  float32
    targets : (N,)                           float32
    """
    enc_x, dec_x, targets = [], [], []
    for i in range(len(series) - seq_len - pred_len + 1):
        enc_seq    = series[i : i + seq_len]
        label_part = series[i + seq_len - label_len : i + seq_len]
        pred_part  = np.zeros(pred_len, dtype=np.float32)
        dec_seq    = np.concatenate([label_part, pred_part])
        target_val = series[i + seq_len]
        enc_x.append(enc_seq)
        dec_x.append(dec_seq)
        targets.append(target_val)
    enc_x   = np.array(enc_x,   dtype=np.float32)[..., np.newaxis]
    dec_x   = np.array(dec_x,   dtype=np.float32)[..., np.newaxis]
    targets = np.array(targets, dtype=np.float32)
    return enc_x, dec_x, targets

In [9]:
class TransformerDataset(Dataset):
    def __init__(self, enc_x, dec_x, targets):
        self.enc_x   = torch.from_numpy(enc_x)
        self.dec_x   = torch.from_numpy(dec_x)
        self.targets = torch.from_numpy(targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.enc_x[idx], self.dec_x[idx], self.targets[idx]

In [10]:
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)                      # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Shared FFN builder ───────────────────────────────────────────────────────
def _make_ff(d_model: int, d_ff: int, dropout: float, activation: str):
    """Two-layer feed-forward sub-layer with specified activation (relu/gelu)."""
    act = nn.ReLU() if activation.lower() == "relu" else nn.GELU()
    return nn.Sequential(
        nn.Linear(d_model, d_ff), act, nn.Dropout(dropout),
        nn.Linear(d_ff, d_model),
    )


# ── Encoder Layer ─────────────────────────────────────────────────────────────
class TransformerEncoderLayer(nn.Module):
    """
    Standard Transformer encoder sub-block.
    Self-attention → Add & Norm → FFN → Add & Norm
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 dropout: float, activation: str):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.ff    = _make_ff(d_model, d_ff, dropout, activation)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


# ── Decoder Layer ─────────────────────────────────────────────────────────────
class TransformerDecoderLayer(nn.Module):
    """
    Standard Transformer decoder sub-block.
    Causal self-attention → Add & Norm → Cross-attention → Add & Norm → FFN → Add & Norm
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 dropout: float, activation: str):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True)
        self.ff    = _make_ff(d_model, d_ff, dropout, activation)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor,
                tgt_mask: torch.Tensor = None) -> torch.Tensor:
        sa_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.drop(sa_out))
        ca_out, _ = self.cross_attn(x, enc_out, enc_out)
        x = self.norm2(x + self.drop(ca_out))
        x = self.norm3(x + self.drop(self.ff(x)))
        return x


# ── Full Vanilla Transformer ───────────────────────────────────────────────────
class VanillaTransformer(nn.Module):
    """
    Encoder-Decoder Transformer for single-step forecasting of raw Close.

    Input shapes:
        enc_x : (B, seq_len,              1)
        dec_x : (B, label_len + pred_len, 1)
    Output:
        (B,)  — one scalar prediction per sample.
    """
    def __init__(self,
                 d_model    : int   = T_D_MODEL,
                 n_heads    : int   = T_N_HEADS,
                 e_layers   : int   = T_E_LAYERS,
                 d_layers   : int   = T_D_LAYERS,
                 d_ff       : int   = T_D_FF,
                 dropout    : float = T_DROPOUT,
                 activation : str   = T_ACTIVATION,
                 seq_len    : int   = T_SEQ_LEN,
                 label_len  : int   = T_LABEL_LEN,
                 pred_len   : int   = T_PRED_LEN):
        super().__init__()
        self.pred_len  = pred_len
        self.label_len = label_len

        # Input projections: univariate (1) → d_model
        self.enc_embed = nn.Linear(1, d_model)
        self.dec_embed = nn.Linear(1, d_model)

        # Positional encodings
        self.enc_pos = PositionalEncoding(
            d_model, max_len=seq_len + 10, dropout=dropout)
        self.dec_pos = PositionalEncoding(
            d_model, max_len=label_len + pred_len + 10, dropout=dropout)

        # Encoder & Decoder stacks
        self.encoder = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, activation)
            for _ in range(e_layers)])
        self.decoder = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, activation)
            for _ in range(d_layers)])

        self.enc_norm = nn.LayerNorm(d_model)
        self.dec_norm = nn.LayerNorm(d_model)

        # Output projection: d_model → scalar
        self.proj = nn.Linear(d_model, 1)

    @staticmethod
    def _causal_mask(sz: int, device: torch.device) -> torch.Tensor:
        """Upper-triangular causal (look-ahead) mask for decoder self-attention."""
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    def forward(self, enc_x: torch.Tensor,
                dec_x: torch.Tensor) -> torch.Tensor:
        # Encoder
        enc_out = self.enc_pos(self.enc_embed(enc_x))
        for layer in self.encoder:
            enc_out = layer(enc_out)
        enc_out = self.enc_norm(enc_out)

        # Decoder
        dec_out = self.dec_pos(self.dec_embed(dec_x))
        tgt_mask = self._causal_mask(dec_out.size(1), dec_out.device)
        for layer in self.decoder:
            dec_out = layer(dec_out, enc_out, tgt_mask)
        dec_out = self.dec_norm(dec_out)

        out = self.proj(dec_out[:, -self.pred_len:, :])   # (B, pred_len, 1)
        return out.squeeze(-1).squeeze(-1)                  # (B,)

In [11]:
def _train_transformer(train_sc : np.ndarray,
                        val_sc   : np.ndarray,
                        test_sc  : np.ndarray,
                        patience : int = T_PATIENCE):
    """
    Train a fresh VanillaTransformer on pre-scaled 1-D sequences.

    Returns
    -------
    model        : trained VanillaTransformer (best weights restored)
    preds_scaled : 1-D float32 numpy array of test-set predictions (scaled)
    val_context  : 1-D array used for seeding recursive multi-step forecast
    """
    # Prepend context to val/test to avoid boundary leakage
    val_context  = np.concatenate([train_sc[-T_SEQ_LEN:], val_sc])
    test_context = np.concatenate([val_sc[-T_SEQ_LEN:],   test_sc])

    enc_tr, dec_tr, y_tr = create_transformer_windows(
        train_sc,    T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)
    enc_vl, dec_vl, y_vl = create_transformer_windows(
        val_context, T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)
    enc_te, dec_te, y_te = create_transformer_windows(
        test_context, T_SEQ_LEN, T_LABEL_LEN, T_PRED_LEN)

    if len(enc_tr) == 0:
        raise ValueError("Training set too short for Transformer windows.")

    pin   = (DEVICE.type == "cuda")
    tr_dl = DataLoader(TransformerDataset(enc_tr, dec_tr, y_tr),
                       T_BATCH, shuffle=False, pin_memory=pin)
    vl_dl = DataLoader(TransformerDataset(enc_vl, dec_vl, y_vl),
                       T_BATCH, shuffle=False)
    te_ds = TransformerDataset(enc_te, dec_te, y_te)

    model  = VanillaTransformer().to(DEVICE)
    opt    = Adam(model.parameters(), lr=T_LR)
    crit   = nn.MSELoss()
    best_v = float("inf")
    best_w = None
    no_imp = 0

    for _epoch in range(1, T_EPOCHS + 1):
        model.train()
        for eb, db, yb in tr_dl:
            eb, db, yb = eb.to(DEVICE), db.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(eb, db), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vl_losses = []
        with torch.no_grad():
            for eb, db, yb in vl_dl:
                vl_losses.append(
                    crit(model(eb.to(DEVICE), db.to(DEVICE)),
                         yb.to(DEVICE)).item())
        vl_loss = float(np.mean(vl_losses))

        if vl_loss < best_v:
            best_v = vl_loss
            best_w = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    # Restore best weights
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_w.items()})
    model.eval()
    te_dl = DataLoader(te_ds, T_BATCH, shuffle=False)
    preds = []
    with torch.no_grad():
        for eb, db, _ in te_dl:
            preds.append(model(eb.to(DEVICE), db.to(DEVICE)).cpu().numpy())

    return model, np.concatenate(preds), val_context   # scaled preds

In [12]:
def recursive_multi_step_forecast(model: nn.Module,
                                   seed_window_scaled: np.ndarray,
                                   steps: int,
                                   scaler: MinMaxScaler) -> np.ndarray:
    """
    Forecast `steps` ahead autoregressively using the Transformer.
    seed_window_scaled : shape (seq_len,) — last `seq_len` scaled values.
    No ground-truth injection during the forecast loop.
    Returns original-scale predictions of length `steps`.
    """
    model.eval()
    curr_enc      = seed_window_scaled.copy().astype(np.float32)  # (seq_len,)
    curr_dec_seed = curr_enc[-T_LABEL_LEN:]                       # (label_len,)

    preds_scaled = []
    with torch.no_grad():
        for _ in range(steps):
            # Encoder tensor: (1, seq_len, 1)
            enc_t = torch.tensor(
                curr_enc, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(DEVICE)
            # Decoder tensor: (1, label_len + pred_len, 1) — last slot masked (zero)
            dec_arr = np.concatenate(
                [curr_dec_seed, np.zeros(T_PRED_LEN, dtype=np.float32)])
            dec_t = torch.tensor(
                dec_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(DEVICE)

            pred_val = model(enc_t, dec_t).item()     # scalar
            preds_scaled.append(pred_val)

            # Slide encoder window by 1
            curr_enc      = np.append(curr_enc[1:], pred_val)
            curr_dec_seed = curr_enc[-T_LABEL_LEN:]

    preds_orig = scaler.inverse_transform(
        np.array(preds_scaled, dtype=np.float32).reshape(-1, 1)).flatten()
    return preds_orig

In [13]:
def display_volatility_regime(df_test: pd.DataFrame,
                               y_true : np.ndarray,
                               y_hat  : np.ndarray):
    """High / low volatility split metrics for the test set (seed=50)."""
    close_all    = df_test[TARGET_COL].values.astype(np.float64)
    log_ret_vals = np.append([np.nan], np.log(close_all[1:] / close_all[:-1]))
    rol_vol      = pd.Series(log_ret_vals).rolling(ROLLING_VOL_WINDOW).std()

    min_len  = min(len(y_true), len(rol_vol))
    vol_vals = rol_vol.values[:min_len]
    y_t      = y_true[:min_len]
    y_h      = y_hat[:min_len]

    vol_med  = float(np.nanmedian(vol_vals))
    vol_vals = np.where(np.isnan(vol_vals), vol_med, vol_vals)

    high_mask = vol_vals >= vol_med
    low_mask  = ~high_mask

    if high_mask.sum() > 0 and low_mask.sum() > 0:
        m_h = compute_metrics(y_t[high_mask], y_h[high_mask])
        m_l = compute_metrics(y_t[low_mask],  y_h[low_mask])

        print("\n" + "=" * 66)
        print("  VOLATILITY REGIME VALIDATION (Seed 50)")
        print(f"  30-day rolling vol | Median = {vol_med:.6f}")
        print("=" * 66)
        print(f'  {"Metric":<12} {"High Volatility":>18} {"Low Volatility":>18}')
        print("-" * 66)
        for k in ["MAE", "RMSE", "MAPE", "R2"]:
            print(f"  {k:<12} {m_h[k]:>18.4f} {m_l[k]:>18.4f}")
        print("=" * 66)
        print(f"  High-vol days : {high_mask.sum()}")
        print(f"  Low-vol  days : {low_mask.sum()}")


In [14]:
def run_experiment(df_train : pd.DataFrame,
                   df_val   : pd.DataFrame,
                   df_test  : pd.DataFrame,
                   seed     : int) -> tuple:
    """
    Full Standalone-Transformer pipeline for one seed:
      1. set_seed (deterministic).
      2. Fit MinMaxScaler on train Close only (no leakage).
      3. Train one VanillaTransformer with early stopping.
      4. Compute test metrics + recursive multi-step robustness metrics.
      5. Return (metrics_dict, y_true, y_hat).
    """
    set_seed(seed)
    t0 = time.time()

    # Fit scaler on training Close only
    scaler   = MinMaxScaler(feature_range=(0, 1))
    train_sc = scaler.fit_transform(
        df_train[[TARGET_COL]].values).flatten().astype(np.float32)
    val_sc   = scaler.transform(
        df_val[[TARGET_COL]].values).flatten().astype(np.float32)
    test_sc  = scaler.transform(
        df_test[[TARGET_COL]].values).flatten().astype(np.float32)

    # Train; get back model + scaled test preds + val context
    model, preds_sc, val_context = _train_transformer(train_sc, val_sc, test_sc)

    t_elapsed = time.time() - t0

    # Inverse-transform predictions and ground-truth
    y_hat  = scaler.inverse_transform(
        preds_sc.reshape(-1, 1)).flatten().astype(np.float64)
    n_pred = len(preds_sc)
    y_true = df_test[TARGET_COL].values.astype(np.float64)[:n_pred]

    # Align lengths
    min_len = min(len(y_hat), len(y_true))
    y_hat   = y_hat[:min_len]
    y_true  = y_true[:min_len]

    metrics = compute_metrics(y_true, y_hat)
    metrics["training_time_sec"] = t_elapsed

    # Robustness: recursive multi-step forecasting (seeded from end of val)
    seed_window = val_context[-T_SEQ_LEN:]
    for step_count in MULTISTEP_HORIZONS:
        multistep_preds = recursive_multi_step_forecast(
            model, seed_window, step_count, scaler)
        mstep_true = scaler.inverse_transform(
            test_sc[:step_count].reshape(-1, 1)).flatten()
        mstep_m = compute_metrics(mstep_true, multistep_preds)
        metrics[f"multi_{step_count}d_MAE"]  = mstep_m["MAE"]
        metrics[f"multi_{step_count}d_RMSE"] = mstep_m["RMSE"]

    return metrics, y_true, y_hat


In [15]:
def print_and_save_summary(all_results: list):
    """Print and save 50-run aggregate statistics."""
    keys = ["MAE", "RMSE", "MAPE", "R2", "training_time_sec"]
    for d in MULTISTEP_HORIZONS:
        keys.extend([f"multi_{d}d_MAE", f"multi_{d}d_RMSE"])

    sep   = "=" * 80
    lines = [
        sep,
        "  STANDALONE TRANSFORMER — 50-RUN STATISTICAL SUMMARY",
        sep,
        f'  {"Metric":<22} {"Mean":>12} {"± Std":>12} {"Min":>10} {"Max":>10}',
        "-" * 80,
    ]
    for k in keys:
        arr = [r[k] for r in all_results]
        mn, sd, mi, mx = np.mean(arr), np.std(arr), np.min(arr), np.max(arr)
        name = "Time (s)" if k == "training_time_sec" else k
        lines.append(f"  {name:<22} {mn:>12.4f} {sd:>12.4f} {mi:>10.4f} {mx:>10.4f}")

    times  = [r["training_time_sec"] for r in all_results]
    mn_t, sd_t = np.mean(times), np.std(times)
    lines += [
        "-" * 80,
        f'  {"Time (min)":<22} {mn_t/60:>12.4f} {sd_t/60:>12.4f}',
        sep,
    ]
    txt = "\n".join(lines)
    print("\n" + txt)

    with open(OUTPUT_TXT, "w") as f:
        f.write(txt + "\n")
    print(f"\nSummary → {OUTPUT_TXT}")
    return keys

In [16]:
def plot_actual_vs_predicted(df_test : pd.DataFrame,
                              y_true  : np.ndarray,
                              y_hat   : np.ndarray):
    """Two-panel chart: price overlay + residual bar."""
    test_dates = df_test["Date"].values[:len(y_true)]

    fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                              gridspec_kw={"height_ratios": [3, 1]})
    ax = axes[0]
    ax.plot(test_dates, y_true, color="#1565C0", lw=1.8, label="Actual Close")
    ax.plot(test_dates, y_hat,  color="#E53935", lw=1.3,
            ls="--", label="Predicted Close")
    ax.set_title("Standalone Transformer: Actual vs Predicted (Test 2023–2025)",
                 fontsize=14, fontweight="bold")
    ax.set_ylabel("NIFTY 50 Close")
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    errs = y_true - y_hat
    ax2.bar(test_dates, errs, color="#7B1FA2", alpha=0.6, width=1)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_title("Prediction Error (Actual − Predicted)")
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Error")
    ax2.grid(True, alpha=0.3)

    m50 = compute_metrics(y_true, y_hat)
    fig.text(0.01, 0.97,
             f'Seed 50  |  MAE={m50["MAE"]:.2f}  RMSE={m50["RMSE"]:.2f}  '
             f'MAPE={m50["MAPE"]:.4f}%  R²={m50["R2"]:.4f}',
             fontsize=10, va="top")
    plt.tight_layout()
    out = "standalone_transformer_actual_vs_predicted.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved → {out}")

In [17]:
def plot_metric_distributions(all_results: list):
    """Four-panel histogram of MAE, RMSE, MAPE, R² across 50 seeds."""
    maes  = [r["MAE"]  for r in all_results]
    rmses = [r["RMSE"] for r in all_results]
    mapes = [r["MAPE"] for r in all_results]
    r2s   = [r["R2"]   for r in all_results]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    data_pairs = [
        ("MAE",      maes,  "#1565C0"),
        ("RMSE",     rmses, "#C62828"),
        ("MAPE (%)", mapes, "#2E7D32"),
        ("R²",       r2s,   "#F57F17"),
    ]
    for ax, (label, arr, col) in zip(axes, data_pairs):
        ax.hist(arr, bins=15, color=col, alpha=0.8, edgecolor="white")
        ax.axvline(np.mean(arr), color="black", ls="--", lw=1.5,
                   label=f"Mean={np.mean(arr):.3f}")
        ax.set_title(label, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        "Metric Distributions across 50 Independent Runs (Standalone Transformer)",
        fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out = "standalone_transformer_metric_distributions.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved → {out}")

In [18]:
def main():
    print(f"Device    : {DEVICE}")
    print(f"PyTorch   : {torch.__version__}")
    print(f"Model     : Standalone Transformer  "
          f"(d_model={T_D_MODEL}, heads={T_N_HEADS}, "
          f"enc={T_E_LAYERS}L, dec={T_D_LAYERS}L, d_ff={T_D_FF})")
    print(f"Seq_len   : {T_SEQ_LEN}  |  Label_len: {T_LABEL_LEN}  |  Pred_len: {T_PRED_LEN}")
    print(f"Activation: {T_ACTIVATION}  |  Dropout: {T_DROPOUT}  |  LR: {T_LR}")
    print(f"Target    : {TARGET_COL} (raw Close — no CEEMDAN)")
    print(f"Split     : ≤2022 (Train/Val 90/10%), ≥2023 (Test)")
    print(f"Runs      : {N_EXPERIMENTS}")
    print("=" * 75)

    # Load & split
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df_train, df_val, df_test = split_data(df)

    print(f"Train : {len(df_train):>5}  "
          f"({df_train.Date.min().date()} → {df_train.Date.max().date()})")
    print(f"Val   : {len(df_val):>5}  "
          f"({df_val.Date.min().date()} → {df_val.Date.max().date()})")
    print(f"Test  : {len(df_test):>5}  "
          f"({df_test.Date.min().date()} → {df_test.Date.max().date()})")
    print("=" * 75)

    all_results  = []
    _last_y_true = None
    _last_y_hat  = None

    for seed in range(1, N_EXPERIMENTS + 1):
        print(f"[{seed:02d}/{N_EXPERIMENTS}] Executing seed={seed}...", end=" ", flush=True)
        m, yt, yp = run_experiment(df_train, df_val, df_test, seed=seed)
        m["experiment"] = seed
        m["seed"]       = seed
        all_results.append(m)
        print(f"Done | MAE={m['MAE']:8.2f} | R²={m['R2']:7.4f} | Time={m['training_time_sec']:.1f}s",
              flush=True)
        _last_y_true = yt
        _last_y_hat  = yp

    print("\n✅ All 50 experiments complete.")

    # Save per-experiment CSV
    keys = ["MAE", "RMSE", "MAPE", "R2", "training_time_sec"]
    for d in MULTISTEP_HORIZONS:
        keys.extend([f"multi_{d}d_MAE", f"multi_{d}d_RMSE"])

    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["experiment", "seed"] + keys,
                                extrasaction="ignore")
        writer.writeheader()
        writer.writerows(all_results)
    print(f"Results saved → {OUTPUT_CSV}")

    # Aggregate summary
    print_and_save_summary(all_results)

    # Plots (seed=50 run)
    plot_actual_vs_predicted(df_test, _last_y_true, _last_y_hat)
    plot_metric_distributions(all_results)

    # Volatility regime validation (seed=50 run)
    display_volatility_regime(df_test, _last_y_true, _last_y_hat)


if __name__ == "__main__":
    main()


Device    : cuda
PyTorch   : 2.10.0+cu128
Model     : Standalone Transformer  (d_model=256, heads=8, enc=2L, dec=1L, d_ff=1024)
Seq_len   : 96  |  Label_len: 48  |  Pred_len: 1
Activation: relu  |  Dropout: 0.1  |  LR: 0.0001
Target    : Close (raw Close — no CEEMDAN)
Split     : ≤2022 (Train/Val 90/10%), ≥2023 (Test)
Runs      : 50
Train :  1770  (2015-01-09 → 2022-03-16)
Val   :   196  (2022-03-17 → 2022-12-30)
Test  :   633  (2023-01-02 → 2025-07-25)
[01/50] Executing seed=1... Done | MAE= 5563.96 | R²=-4.6743 | Time=21.8s
[02/50] Executing seed=2... Done | MAE= 3708.95 | R²=-1.8691 | Time=12.4s
[03/50] Executing seed=3... Done | MAE= 3109.55 | R²=-1.0032 | Time=16.0s
[04/50] Executing seed=4... Done | MAE= 5766.79 | R²=-5.0640 | Time=13.5s
[05/50] Executing seed=5... Done | MAE= 3351.50 | R²=-1.3660 | Time=12.9s
[06/50] Executing seed=6... Done | MAE= 2084.93 | R²= 0.0858 | Time=32.6s
[07/50] Executing seed=7... Done | MAE= 4379.26 | R²=-2.7330 | Time=13.4s
[08/50] Executing seed=8